# Retrieval Models — ALS, Item2Vec, LightGCN, GraphSAGE, Two-Tower

**Pipeline**: Train model → Extract embeddings → FAISS index (small_matrix items only) → Evaluate

**Eval protocol**: KuaiRec unbiased evaluation — rank only among 3,327 small_matrix videos (99.6% density = near-complete ground truth).

**Adaptive labeling**: Threshold varies by video duration — short videos need higher `watch_ratio`, long videos need lower.

All logic lives in `src/` modules. This notebook is a thin training & benchmark runner.

## 0. Setup

In [ ]:
import os, sys

# --- Colab detection ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to your repo location on Google Drive
    REPO_ROOT = '/content/drive/MyDrive/Final-Project-Bootcamp-Recsys'
    os.chdir(REPO_ROOT)
    !pip install -q implicit gensim faiss-cpu torch-geometric
else:
    REPO_ROOT = os.path.abspath('..')
    os.chdir(REPO_ROOT)

sys.path.insert(0, REPO_ROOT)
print(f'Working directory: {os.getcwd()}')
print(f'Colab: {IN_COLAB}')

In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path

sns.set_theme(style='whitegrid')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)
EXP_DIR = Path('experiments')
EXP_DIR.mkdir(exist_ok=True)

print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')

## 1. Load Data & Ground Truth

In [ ]:
from src.data.preprocessing import (
    load_big_matrix, load_small_matrix, split_big_matrix,
    build_ground_truth, build_adaptive_ground_truth,
)
from src.evaluation.retrieval_eval import evaluate_via_faiss
from src.indexing.faiss_index import FAISSIndex

big   = load_big_matrix()
small = load_small_matrix()

train, val, test = split_big_matrix(big)
del big  # free ~1GB
gc.collect()

# Ground truths
gt_fixed    = build_ground_truth(small)
gt_adaptive = build_adaptive_ground_truth(small)

candidate_pool = set(small['video_id'].unique())
print(f'Candidate pool: {len(candidate_pool):,} videos')

## 2. Helper: extract embeddings & evaluate

In [ ]:
# Collector for all results
all_scores = {}   # {model_name: scores_dict}
all_recs   = {}   # {model_name: {user_id: [video_ids]}}
K_LIST = [10, 20, 50, 100]


def eval_model(model_name, item_ids, item_embs, user_ids, user_embs):
    """Evaluate and store results for a model."""
    scores, recs = evaluate_via_faiss(
        item_ids, item_embs, user_ids, user_embs,
        gt_adaptive, candidate_pool, model_name, k_list=K_LIST,
    )
    all_scores[model_name] = scores
    all_recs[model_name]   = recs
    return scores, recs

---
## 3. ALS (Week 1 baseline)

In [ ]:
from src.retrieval.als import ALSRetriever

als = ALSRetriever(factors=128, iterations=30, alpha=40)
als.fit(train)
als.save(str(MODEL_DIR / 'als.pkl'))

In [ ]:
als_item_ids = [als.idx2item[i] for i in range(len(als.idx2item))]
als_item_embs = als.get_item_embeddings()
als_user_ids  = [als.idx2user[i] for i in range(len(als.idx2user))]
als_user_embs = als.get_user_embeddings()

eval_model('ALS', als_item_ids, als_item_embs, als_user_ids, als_user_embs)

---
## 4. Item2Vec (Week 1 baseline)

In [ ]:
from src.retrieval.item2vec import Item2VecRetriever

i2v = Item2VecRetriever(vector_size=128, window=50, epochs=10)
i2v.fit(train)
i2v.save(str(MODEL_DIR / 'item2vec.pkl'))

In [ ]:
i2v_item_ids, i2v_item_embs = i2v.get_item_embeddings()
i2v_user_ids = list(i2v.user_embeddings.keys())
i2v_user_embs = np.stack([i2v.user_embeddings[u] for u in i2v_user_ids])

eval_model('Item2Vec', i2v_item_ids, i2v_item_embs, i2v_user_ids, i2v_user_embs)

---
## 5. LightGCN (Week 2)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from src.retrieval.lightgcn import LightGCNRetriever

lgcn = LightGCNRetriever(
    embedding_dim=128, n_layers=3,
    epochs=150, patience=15, batch_size=4096,
)
lgcn.fit(train)
lgcn.save(str(MODEL_DIR / 'lightgcn.pkl'))

In [ ]:
lgcn_item_ids, lgcn_item_embs = lgcn.get_item_embeddings()
lgcn_user_ids, lgcn_user_embs = lgcn.get_user_embeddings()

eval_model('LightGCN', lgcn_item_ids, lgcn_item_embs, lgcn_user_ids, lgcn_user_embs)

---
## 6. GraphSAGE (Week 2)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from src.retrieval.graphsage import GraphSAGERetriever

gsage = GraphSAGERetriever(
    embedding_dim=128, hidden_dim=128, n_layers=2,
    epochs=100, patience=15, batch_size=4096,
)
gsage.fit(train)
gsage.save(str(MODEL_DIR / 'graphsage.pkl'))

In [ ]:
gsage_item_ids, gsage_item_embs = gsage.get_item_embeddings()
gsage_user_ids, gsage_user_embs = gsage.get_user_embeddings()

eval_model('GraphSAGE', gsage_item_ids, gsage_item_embs, gsage_user_ids, gsage_user_embs)

---
## 7. Two-Tower (Week 2)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from src.retrieval.two_tower import TwoTowerRetriever

tt = TwoTowerRetriever(
    embedding_dim=128, hidden_dim=256,
    epochs=50, patience=10, batch_size=4096,
)
tt.fit(train)
tt.save(str(MODEL_DIR / 'two_tower.pkl'))

In [ ]:
tt_item_ids, tt_item_embs = tt.get_item_embeddings()
tt_user_ids, tt_user_embs = tt.get_user_embeddings()

eval_model('Two-Tower', tt_item_ids, tt_item_embs, tt_user_ids, tt_user_embs)

---
## 8. Comparison Table

In [ ]:
results_df = pd.DataFrame(all_scores).T
results_df.index.name = 'Model'
col_order = [f'{m}@{k}' for m in ['NDCG', 'Recall'] for k in K_LIST]
results_df = results_df[[c for c in col_order if c in results_df.columns]].round(4)

print(results_df.to_string())
results_df.to_csv(str(EXP_DIR / 'retrieval_results.csv'))
print(f'\nSaved to {EXP_DIR / "retrieval_results.csv"}')

In [ ]:
# Recall@K and NDCG@K curves (adaptive ground truth)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name, scores in all_scores.items():
    axes[0].plot(K_LIST, [scores[f'Recall@{k}'] for k in K_LIST], marker='o', label=model_name)
    axes[1].plot(K_LIST, [scores[f'NDCG@{k}']   for k in K_LIST], marker='o', label=model_name)

axes[0].set_title('Recall@K (adaptive)'); axes[0].set_xlabel('K'); axes[0].legend()
axes[1].set_title('NDCG@K (adaptive)');   axes[1].set_xlabel('K'); axes[1].legend()
plt.tight_layout()
plt.savefig(str(EXP_DIR / 'retrieval_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 9. FAISS Latency Benchmark

In [ ]:
# Pick the best retrieval model for FAISS benchmark
best_model = max(all_scores, key=lambda m: all_scores[m].get('NDCG@10', 0))
print(f'Best model: {best_model} (NDCG@10={all_scores[best_model]["NDCG@10"]:.4f})')

# Use ALS embeddings for latency benchmark (full catalog)
idx_exact = FAISSIndex(mode='exact')
idx_exact.build(als_item_ids, als_item_embs)
bench_exact = idx_exact.benchmark(als_user_embs, k=100, n_queries=200)

idx_approx = FAISSIndex(mode='approx', n_list=100)
idx_approx.build(als_item_ids, als_item_embs)
bench_approx = idx_approx.benchmark(als_user_embs, k=100, n_queries=200)

print(f"\nExact  — mean: {bench_exact['latency_ms_mean']:.2f}ms  p95: {bench_exact['latency_ms_p95']:.2f}ms")
print(f"Approx — mean: {bench_approx['latency_ms_mean']:.2f}ms  p95: {bench_approx['latency_ms_p95']:.2f}ms")
print('Target: < 10ms per query')

---
## 10. Load saved models (demo)

After training on Colab, download the `models/` folder. Models can be loaded without retraining:

In [ ]:
# Example: load saved models and extract embeddings
# (uncomment to use)

# from src.retrieval.als import ALSRetriever
# als = ALSRetriever.load('models/als.pkl')

# from src.retrieval.item2vec import Item2VecRetriever
# i2v = Item2VecRetriever.load('models/item2vec.pkl')

# from src.retrieval.lightgcn import LightGCNRetriever
# lgcn = LightGCNRetriever.load('models/lightgcn.pkl')

# from src.retrieval.graphsage import GraphSAGERetriever
# gsage = GraphSAGERetriever.load('models/graphsage.pkl')

# from src.retrieval.two_tower import TwoTowerRetriever
# tt = TwoTowerRetriever.load('models/two_tower.pkl')

print('All models can be loaded from models/ directory.')